## 📓 Notebook 03: GridSearchCV vs. RandomizedSearchCV

**Skills practiced:** Defining a hyperparameter space, `GridSearchCV`, `RandomizedSearchCV`, timing searches.

**Goal:** Run both searches on the same space and compare combinations tried, wall-clock time, and best CV score.

In [46]:
import pandas as pd
import numpy as np
import time
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [47]:
x=pd.read_csv('../data/processed/x.csv')
y=pd.read_csv('../data/processed/y.csv').squeeze()

In [48]:
results = []
grid_start_time = time.time()
m=xgb.XGBClassifier(random_state=42, eval_metric='auc')
params={
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}
GS=GridSearchCV(m, params, cv=StratifiedKFold(n_splits=5, shuffle=True, 
                                              random_state=42), scoring='roc_auc', n_jobs=-1)
for train, test in GS.cv.split(x, y):
    GS.fit(x.iloc[train], y.iloc[train])
    GS_pred=GS.predict_proba(x.iloc[test])[:,1]
    GS_roc_auc=roc_auc_score(y.iloc[test], GS_pred)
    results.append(round(GS_roc_auc, 4))
grid_time = time.time() - grid_start_time
print(pd.DataFrame(GS.cv_results_))

    mean_fit_time  std_fit_time  ...  std_test_score  rank_test_score
0        0.780438      0.054929  ...        0.012133               11
1        1.466134      0.085325  ...        0.013439                4
2        2.133168      0.066718  ...        0.013016                1
3        0.934793      0.041602  ...        0.011084                7
4        1.671984      0.034885  ...        0.011239                5
5        2.817952      0.167266  ...        0.011579                3
6        1.350100      0.257102  ...        0.009864               15
7        3.119629      0.197855  ...        0.009928               13
8        3.820191      0.389969  ...        0.009702               12
9        0.817555      0.023485  ...        0.012796                2
10       1.429768      0.084389  ...        0.012235                6
11       2.612636      0.318059  ...        0.011640                8
12       1.326943      0.218329  ...        0.011775                9
13       2.799824   

In [49]:
random_start_time = time.time()
results2 = []
m=xgb.XGBClassifier(random_state=42, eval_metric='auc')
params={
    'n_estimators': range(100, 800, 10),  # RandomizedSearchCV can handle random distributions
    'max_depth': range(2, 15, 1),  # RandomizedSearchCV can handle ranges
    'learning_rate': np.random.uniform(0.005, 0.4, 100)  # RandomizedSearchCV can handle random distributions
}
RS=RandomizedSearchCV(m, params, n_iter=30, cv=StratifiedKFold(n_splits=5, shuffle=True, 
                                                    random_state=42), scoring='roc_auc', n_jobs=-1)
for train, test in RS.cv.split(x, y):
    RS.fit(x.iloc[train], y.iloc[train])
    RS_pred=RS.predict_proba(x.iloc[test])[:,1]
    RS_roc_auc=roc_auc_score(y.iloc[test], RS_pred)
    results2.append(round(RS_roc_auc, 4))
random_time = time.time() - random_start_time
print(pd.DataFrame(RS.cv_results_))

    mean_fit_time  std_fit_time  ...  std_test_score  rank_test_score
0        8.807570      0.372969  ...        0.009662               17
1        6.826906      0.357896  ...        0.009969               30
2        4.883790      0.083735  ...        0.009572               25
3        4.682480      0.248190  ...        0.011191               19
4        4.814369      0.051468  ...        0.012663               14
5        8.800405      0.198439  ...        0.009594               22
6        2.402281      0.182996  ...        0.010141                6
7        4.371374      0.399628  ...        0.010209               10
8        0.991147      0.053789  ...        0.010780                5
9        1.076952      0.066813  ...        0.010293                8
10       4.552878      0.113392  ...        0.010651               11
11       5.804863      1.173975  ...        0.011032                9
12       7.650433      0.398326  ...        0.010562               28
13       4.882743   

In [50]:
print(f"Best parameters from Grid Search: \n{GS.best_params_}")
print(f"Best parameters from Randomized Search: \n{RS.best_params_}")
print()
print(f"{GS.best_score_:.4f} is the best ROC AUC score from Grid Search")
print(f"{RS.best_score_:.4f} is the best ROC AUC score from Randomized Search")
print()
print(f"Time taken for Grid Search: {grid_time:.2f} seconds")
print(f"Time taken for Randomized Search: {random_time:.2f} seconds")
print()
print("ROC AUC scores Comparison Table:")
tb=pd.DataFrame({
    'Grid Search': results,
    'Randomized Search': results2
})
print(tb)

Best parameters from Grid Search: 
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
Best parameters from Randomized Search: 
{'n_estimators': 410, 'max_depth': 2, 'learning_rate': np.float64(0.09128485301851597)}

0.8479 is the best ROC AUC score from Grid Search
0.8446 is the best ROC AUC score from Randomized Search

Time taken for Grid Search: 195.85 seconds
Time taken for Randomized Search: 558.41 seconds

ROC AUC scores Comparison Table:
   Grid Search  Randomized Search
0       0.8600             0.8622
1       0.8443             0.8433
2       0.8552             0.8562
3       0.8305             0.8311
4       0.8389             0.8373
